In [3]:
import pandas as pd
import numpy as np

path_predictions = ["../results/cta_rsna_ane/cnn_4l_input_edt_amp_ogopt/inference_final/predict.csv", "../results/cta_rsna_ane/aug_cnnfa_4l_short_input_edt_fp_strict_fix/inference_final/predict.csv", "../results/cta_rsna_ane/opaug_cnn_4l_short_input_edt/inference_final/predict.csv" ]
path_annot = "/scratch/ceballosarroyo.a/aneurysm/mm_datasets/cta_rsna_ane/annotations.csv"
df_annot = pd.read_csv(path_annot)
headers = ["idx", "seriesuid", "coordZ", "coordY", "coordX", "d", "h", "w", "probability"]


In [6]:
print(len(df_annot))
df_annot = df_annot[df_annot["w"] < 300]
print(len(df_annot))


1149
1065


In [7]:
def overlaps(z,y,x, z_pred, y_pred, x_pred, d, h, w):
    z_pred_0 = z_pred - d/2
    z_pred_1 = z_pred + d/2
    y_pred_0 = y_pred - h/2
    y_pred_1 = y_pred + h/2
    x_pred_0 = x_pred - w/2
    x_pred_1 = x_pred + w/2
    # verify if z,y,x is inside the predicted box
    if (z >= z_pred_0 and z <= z_pred_1) and (y >= y_pred_0 and y <= y_pred_1) and (x >= x_pred_0 and x <= x_pred_1):
        
        return True

    else:
        return False


In [8]:
import tqdm
values = []
has_pred = []
for i, row in tqdm.tqdm(df_annot.iterrows()):
    seriesuid = row["seriesuid"]
    z,y,x = row["coordZ"], row["coordY"], row["coordX"]
    
    for path_pred in path_predictions:
        df_pred = pd.read_csv(path_pred)
        pred_row = df_pred[df_pred["seriesuid"] == seriesuid]
        # check 
        z_pred, y_pred, x_pred, d, h, w = pred_row[["coordZ", "coordY", "coordX", "d", "h", "w"]].values[0]
        # check if there is overlap 
        over = overlaps(z,y,x, z_pred, y_pred, x_pred, d, h, w)
        if over:
            values.append([i, seriesuid, z_pred, y_pred, x_pred, d, h, w, pred_row["probability"].values[0]])
            has_pred.append(i)
        

0it [00:00, ?it/s]

1065it [01:51,  9.59it/s]


In [9]:
has_pred = set(has_pred)
print(f"Number of annotations with predictions: {len(has_pred)}/{len(df_annot)}")

Number of annotations with predictions: 843/1065


In [10]:
843/1065

0.7915492957746478

In [11]:
df_new_headers = df_annot.columns 

rows = []

all_d = [v[5] for v in values]
all_h = [v[6] for v in values]
all_w = [v[7] for v in values]

count = 0
for idx, row in df_annot.iterrows():
    vals = [v for v in values if v[0] == idx]
    sample_d = [v[5] for v in vals]
    sample_h = [v[6] for v in vals]
    sample_w = [v[7] for v in vals]
    proba = [v[8] for v in vals]

    for i_p, p in enumerate(proba):
        if p < 0.5:
            sample_d.pop(i_p)
            sample_h.pop(i_p)
            sample_w.pop(i_p)

    if len(sample_d) > 0:
        mean_d = np.mean(sample_d)
        mean_h = np.mean(sample_h)
        mean_w = np.mean(sample_w)
        count += 1
    else:
        mean_d = np.mean(all_d)
        mean_h = np.mean(all_h)
        mean_w = np.mean(all_w)
    
    volume = mean_d * mean_h * mean_w
    major_axis = max(mean_d, mean_h, mean_w)
    minor_axis = min(mean_d, mean_h, mean_w)

    
    new_row = [
                row["seriesuid"],row["coordX"], row["coordY"], row["coordZ"], mean_d, mean_h, mean_w, "aneurysm", volume, major_axis, minor_axis
               ]
    
    rows.append(new_row)


df_annot_with_sizes = pd.DataFrame(rows, columns=["seriesuid","coordX", "coordY", "coordZ", "d", "h", "w", "label", "volume", "major_axis", "minor_axis"])


In [12]:
len(df_annot_with_sizes) - count 

222

In [13]:
df_annot_with_sizes.to_csv("../labels/gt/cta_rsna_ane_annotations2.csv", index=False)

list_uids = df_annot_with_sizes['seriesuid'].unique().tolist()

import pickle 

with open('../labels/gt/cta_rsna_ane_seriesuids.pkl', 'wb') as f:
    pickle.dump(list_uids, f)

In [ ]:
# remove files recursively
